# Company-Only Adapter Training Notebook

This notebook is scoped to company-domain baseline + training only.
1. Baseline evaluation on company data
2. Fine-tune a company adapter
3. Compare baseline vs company adapter

## 1) Install Dependencies (Colab)

Run this once per fresh runtime.

In [1]:
!pip -q install --upgrade pip
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers datasets accelerate bitsandbytes peft trl sentencepiece pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.0 MB/s eta 0:00:00


In [2]:
import os
import shutil
import subprocess
import time
from pathlib import Path

REPO_URL = "https://github.com/Arul-7781/QueryMate.git"
REPO_BRANCH = "validation"
FALLBACK_BRANCH = "main"
REPO_DIR = Path('/content/QueryMate')


def run_cmd(cmd, check=True):
    print('$', ' '.join(cmd))
    res = subprocess.run(cmd, text=True, capture_output=True)
    if res.stdout.strip():
        print(res.stdout.strip())
    if res.returncode != 0:
        if res.stderr.strip():
            print(res.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed ({res.returncode}): {' '.join(cmd)}")
    return res


token = os.getenv('GITHUB_TOKEN', '').strip()
auth_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL


def branch_exists(url, branch):
    res = subprocess.run(['git', 'ls-remote', '--heads', url, branch], text=True, capture_output=True)
    return res.returncode == 0 and bool(res.stdout.strip())


target_branch = REPO_BRANCH if branch_exists(auth_url, REPO_BRANCH) else FALLBACK_BRANCH
if target_branch != REPO_BRANCH:
    print(f"Warning: branch '{REPO_BRANCH}' not found. Falling back to '{FALLBACK_BRANCH}'.")

if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    print(f"Removing stale directory: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    clone_cmd = ['git', 'clone', '--branch', target_branch, '--single-branch', auth_url, str(REPO_DIR)]
    last_error = None
    for attempt in range(1, 4):
        print(f"Clone attempt {attempt}/3")
        res = subprocess.run(clone_cmd, text=True, capture_output=True)
        if res.returncode == 0:
            if res.stdout.strip():
                print(res.stdout.strip())
            break
        last_error = res.stderr.strip() or res.stdout.strip()
        print(last_error)
        time.sleep(2 * attempt)
    else:
        raise RuntimeError(f"Clone failed after 3 attempts. Last error: {last_error}")
else:
    run_cmd(['git', '-C', str(REPO_DIR), 'fetch', 'origin', target_branch], check=False)
    run_cmd(['git', '-C', str(REPO_DIR), 'checkout', target_branch])
    run_cmd(['git', '-C', str(REPO_DIR), 'pull', 'origin', target_branch], check=False)

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
print('Top-level files:', sorted(os.listdir('.'))[:20])

Clone attempt 1/3
Working directory: /content/QueryMate
Top-level files: ['.DS_Store', '.git', '.gitignore', 'README.md', 'app.py', 'data', 'docs', 'fetch_papers.py', 'notebooks', 'requirements.txt', 'scripts', 'src', 'tests']


## 1.5) Clone QueryMate Repo (Same as First Notebook)

Set `REPO_URL` and branch. This keeps Colab behavior consistent with your original qwen notebook flow.

## 2) Setup Imports, Seeds, and Config

In [3]:
import gc
import os
import re
import sys
import json
import math
import time
import random
import sqlite3
import inspect
import importlib.util
from pathlib import Path
from dataclasses import dataclass
from collections import Counter, defaultdict

import torch
import pandas as pd
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"

# Prefer cloned repo path when available.
if "REPO_DIR" in globals() and Path(REPO_DIR).exists():
    REPO_ROOT = Path(REPO_DIR)
else:
    REPO_ROOT = Path("/content/QueryMate") if Path("/content/QueryMate").exists() else Path.cwd()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

TESTS_DIR = REPO_ROOT / "tests"
DATA_DIR = REPO_ROOT / "data"
OUTPUT_ROOT = Path("/content/outputs/dual_adapters")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

COMPANY_GOLDEN_PATH = TESTS_DIR / "golden_set1.json"
COMPANY_DB_PATH = DATA_DIR / "company_expanded.db"
if not COMPANY_DB_PATH.exists():
    COMPANY_DB_PATH = DATA_DIR / "company.db"

# Dataset size knobs for faster or fuller runs.
COMPANY_MAX_TRAIN = None
COMPANY_MAX_VAL = 150
COMPANY_MAX_TEST = 50
SPIDER_MAX_TRAIN = 3000
SPIDER_MAX_VAL = 200
SPIDER_MAX_TEST = 50

BASELINE_EVAL_COMPANY_N = 50
BASELINE_EVAL_SPIDER_N = 50
FT_EVAL_COMPANY_N = 50
FT_EVAL_SPIDER_N = 50

MAX_SEQ_LEN = 1024
MAX_NEW_TOKENS = 128

print("Repo root:", REPO_ROOT)
print("Company golden:", COMPANY_GOLDEN_PATH)
print("Company DB:", COMPANY_DB_PATH)
print("Output root:", OUTPUT_ROOT)
print("CUDA available:", torch.cuda.is_available())

Repo root: /content/QueryMate
Company golden: /content/QueryMate/tests/golden_set1.json
Company DB: /content/QueryMate/data/company_expanded.db
Output root: /content/outputs/dual_adapters
CUDA available: True


## 3) Load Company Dataset (`golden_set1`) and Create Splits

In [4]:
if not COMPANY_GOLDEN_PATH.exists():
    raise FileNotFoundError(f"Missing company dataset: {COMPANY_GOLDEN_PATH}")

with COMPANY_GOLDEN_PATH.open("r") as f:
    company_cases_all = json.load(f)


def stratified_split(cases, seed=SEED):
    rng = random.Random(seed)
    groups = defaultdict(list)
    for c in cases:
        key = (c.get("difficulty", "medium"), c.get("category", "generated"))
        groups[key].append(c)

    train, val, test = [], [], []
    for _, items in groups.items():
        rng.shuffle(items)
        n = len(items)
        n_train = max(1, int(0.7 * n)) if n >= 3 else max(1, n - 1)
        n_val = int(0.15 * n) if n >= 8 else (1 if n >= 5 else 0)
        if n_train + n_val >= n:
            n_val = max(0, n - n_train - 1)
        n_test = n - n_train - n_val

        train.extend(items[:n_train])
        val.extend(items[n_train:n_train + n_val])
        test.extend(items[n_train + n_val:n_train + n_val + n_test])

    rng.shuffle(train)
    rng.shuffle(val)
    rng.shuffle(test)
    return train, val, test


company_train, company_val, company_test = stratified_split(company_cases_all)

if COMPANY_MAX_TEST and len(company_test) > COMPANY_MAX_TEST:
    overflow_test = company_test[COMPANY_MAX_TEST:]
    company_test = company_test[:COMPANY_MAX_TEST]

    # Reassign removed test rows to validation first (up to cap), then training.
    if COMPANY_MAX_VAL:
        val_room = max(COMPANY_MAX_VAL - len(company_val), 0)
        company_val.extend(overflow_test[:val_room])
        overflow_test = overflow_test[val_room:]

    company_train.extend(overflow_test)

if COMPANY_MAX_TRAIN:
    company_train = company_train[:COMPANY_MAX_TRAIN]
if COMPANY_MAX_VAL:
    company_val = company_val[:COMPANY_MAX_VAL]

rebalance_rng = random.Random(SEED + 7)
rebalance_rng.shuffle(company_train)
rebalance_rng.shuffle(company_val)
rebalance_rng.shuffle(company_test)

print("Company total:", len(company_cases_all))
print("Company split sizes:", {"train": len(company_train), "val": len(company_val), "test": len(company_test)})

Company total: 1000
Company split sizes: {'train': 800, 'val': 150, 'test': 50}


## 5) Utility Functions: Schema, Prompting, SQL Parsing, Runtime, and Evaluation

In [5]:
def load_company_schema_text(db_path: Path) -> str:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("SELECT name, sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name")
    rows = cur.fetchall()
    conn.close()
    return "\n\n".join([f"{name}: {sql}" for name, sql in rows if sql])


COMPANY_SCHEMA_TEXT = load_company_schema_text(COMPANY_DB_PATH)


def canonical_sql(sql: str) -> str:
    if not sql:
        return ""
    s = sql.strip().strip("`")
    s = re.sub(r"\s+", " ", s)
    return s.lower().rstrip(";")


def extract_sql(text: str) -> str:
    if text is None:
        return ""
    t = text.strip()
    t = t.replace("```sql", "```")
    if "```" in t:
        parts = t.split("```")
        if len(parts) >= 2:
            t = parts[1].strip()

    m = re.search(r"(?is)(with|select)\b[\s\S]*", t)
    if m:
        t = m.group(0).strip()

    # keep the first statement
    if ";" in t:
        t = t.split(";")[0].strip() + ";"
    else:
        t = t.strip() + ";"
    return t


def normalise_rows(rows):
    out = []
    for row in rows:
        norm_row = []
        for v in row:
            if isinstance(v, float):
                norm_row.append(round(v, 6))
            else:
                norm_row.append(v)
        out.append(tuple(norm_row))
    return Counter(out)


def execute_sql(db_path: Path, sql: str):
    try:
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        conn.close()
        return rows, None
    except Exception as e:
        return None, str(e)


def spider_schema_text(ex: dict) -> str:
    db_id = ex.get("db_id", "unknown_db")
    tnames = ex.get("table_names_original") or ex.get("table_names") or []
    cnames = ex.get("column_names_original") or ex.get("column_names") or []

    table_to_cols = defaultdict(list)
    for col in cnames:
        if not isinstance(col, (list, tuple)) or len(col) < 2:
            continue
        tidx, cname = col[0], col[1]
        if isinstance(tidx, int) and tidx >= 0 and tidx < len(tnames):
            table_to_cols[tnames[tidx]].append(cname)

    lines = [f"DB: {db_id}"]
    for t in tnames:
        cols = table_to_cols.get(t, [])
        lines.append(f"{t}({', '.join(cols)})")
    return "\n".join(lines)


def company_prompt(question: str) -> str:
    return (
        "You are an expert SQLite SQL assistant. "
        "Generate only SQL with no explanation.\n\n"
        f"Question: {question}\n\n"
        f"Schema:\n{COMPANY_SCHEMA_TEXT}\n\n"
        "SQL:"
    )


def spider_prompt(ex: dict) -> str:
    q = ex.get("question", "")
    schema_txt = spider_schema_text(ex)
    return (
        "You are an expert SQL assistant for text-to-SQL. "
        "Generate only SQL with no explanation.\n\n"
        f"Question: {q}\n\n"
        f"Schema:\n{schema_txt}\n\n"
        "SQL:"
    )


def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_runtime(model_id=BASE_MODEL_ID, adapter_path=None):
    clear_cuda()

    compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    if adapter_path is not None:
        model = PeftModel.from_pretrained(model, adapter_path)

    model.eval()
    return {"model": model, "tokenizer": tokenizer, "model_id": model_id, "adapter_path": adapter_path}


def generate_sql(runtime, prompt: str, temperature: float = 0.0, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    tokenizer = runtime["tokenizer"]
    model = runtime["model"]

    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": "Return only SQL."},
            {"role": "user", "content": prompt},
        ]
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        input_text = prompt

    inputs = tokenizer(input_text, return_tensors="pt")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 0.01) if temperature > 0 else None,
            top_p=0.95 if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = out[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return extract_sql(text)


def evaluate_company_execution(runtime, samples, db_path: Path):
    passed, failed, errors = 0, 0, 0
    details = []

    for s in samples:
        q = s["question"]
        expected_sql = s["sql"]
        pred_sql = generate_sql(runtime, company_prompt(q), temperature=0.0)

        exp_rows, exp_err = execute_sql(db_path, expected_sql)
        pred_rows, pred_err = execute_sql(db_path, pred_sql)

        if exp_err is not None:
            errors += 1
            details.append({"question": q, "status": "gold_error", "pred_sql": pred_sql, "error": exp_err})
            continue

        if pred_err is not None:
            failed += 1
            details.append({"question": q, "status": "pred_error", "pred_sql": pred_sql, "error": pred_err})
            continue

        match = normalise_rows(exp_rows) == normalise_rows(pred_rows)
        if match:
            passed += 1
            status = "pass"
        else:
            failed += 1
            status = "fail"

        details.append({"question": q, "status": status, "pred_sql": pred_sql})

    evaluated = max(1, passed + failed)
    return {
        "total": len(samples),
        "passed": passed,
        "failed": failed,
        "errors": errors,
        "execution_accuracy": passed / evaluated,
        "details": details,
    }


def evaluate_spider_exact(runtime, samples):
    passed, failed = 0, 0
    details = []

    for ex in samples:
        q = ex.get("question", "")
        gold_sql = ex.get("query", "")
        pred_sql = generate_sql(runtime, spider_prompt(ex), temperature=0.0)

        match = canonical_sql(gold_sql) == canonical_sql(pred_sql)
        if match:
            passed += 1
            status = "pass"
        else:
            failed += 1
            status = "fail"

        details.append({
            "db_id": ex.get("db_id", ""),
            "question": q,
            "status": status,
            "gold_sql": gold_sql,
            "pred_sql": pred_sql,
        })

    total = max(1, passed + failed)
    return {
        "total": passed + failed,
        "passed": passed,
        "failed": failed,
        "exact_match": passed / total,
        "details": details,
    }


print("Utility functions loaded.")

Utility functions loaded.


In [6]:
# Override with a safer generation helper (avoids passing None generation args).
def generate_sql(runtime, prompt: str, temperature: float = 0.0, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    tokenizer = runtime["tokenizer"]
    model = runtime["model"]

    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": "Return only SQL."},
            {"role": "user", "content": prompt},
        ]
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        input_text = prompt

    inputs = tokenizer(input_text, return_tensors="pt")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = {k: v.to(device) for k, v in inputs.items()}

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": temperature > 0,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if temperature > 0:
        gen_kwargs["temperature"] = max(temperature, 0.01)
        gen_kwargs["top_p"] = 0.95

    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)

    gen_tokens = out[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return extract_sql(text)

print("Generation helper ready.")

Generation helper ready.


In [7]:
import types


def install_querymate_compat_shims():
    # Provide provider-free shims so repo imports work in local Qwen-only runs.
    if "src.llm_engine" not in sys.modules:
        llm_mod = types.ModuleType("src.llm_engine")

        def _disabled_provider_invoke(prompt, temperature=0.0):
            raise RuntimeError(
                "External provider invoke_with_fallback is disabled in this notebook runtime. "
                "Use patch_agents_with_runtime(...) to route generation through local Qwen runtime."
            )

        llm_mod.invoke_with_fallback = _disabled_provider_invoke
        sys.modules["src.llm_engine"] = llm_mod

    if "src.schema_rag" not in sys.modules:
        rag_mod = types.ModuleType("src.schema_rag")

        def _init_schema_vectorstore_noop():
            return None

        def _retrieve_relevant_schemas_fallback(question, top_k=3):
            schema_txt = globals().get("COMPANY_SCHEMA_TEXT", "")
            if schema_txt:
                return (
                    "Relevant Tables:\n\n"
                    f"{schema_txt}\n\n"
                    "Relationships:\n"
                    "- Employees.DeptID links to Departments.DeptID\n"
                    "- Projects.DeptID links to Departments.DeptID"
                )
            return "Relevant Tables:\n(Company schema unavailable in fallback mode.)"

        rag_mod.initialize_schema_vectorstore = _init_schema_vectorstore_noop
        rag_mod.retrieve_relevant_schemas = _retrieve_relevant_schemas_fallback
        sys.modules["src.schema_rag"] = rag_mod


install_querymate_compat_shims()

# Load evaluator from repo path (avoids import collisions with unrelated `tests` packages).
EVALUATOR_PATH = REPO_ROOT / "tests" / "evaluator.py"
if not EVALUATOR_PATH.exists():
    raise FileNotFoundError(f"Missing evaluator at: {EVALUATOR_PATH}")

spec_eval = importlib.util.spec_from_file_location("querymate_evaluator", EVALUATOR_PATH)
evaluator = importlib.util.module_from_spec(spec_eval)
if spec_eval is None or spec_eval.loader is None:
    raise RuntimeError("Could not load QueryMate evaluator module")
spec_eval.loader.exec_module(evaluator)

evaluator.DB_PATH = str(COMPANY_DB_PATH)
evaluator.GOLDEN_SET = str(COMPANY_GOLDEN_PATH)
evaluator.SLEEP_BETWEEN = 0.0


@dataclass
class LocalResponse:
    content: str


class AgenticRuntime:
    def __init__(self, runtime_dict):
        self.runtime_dict = runtime_dict

    def invoke(self, prompt, temperature=0.0):
        sql = generate_sql(self.runtime_dict, prompt, temperature=temperature)
        return LocalResponse(content=sql)


def patch_agents_with_runtime(runtime_obj):
    import src.agents as agents

    agents._ACTIVE_RUNTIME = runtime_obj

    def local_invoke_with_fallback(prompt, temperature=0.0):
        active_runtime = getattr(agents, "_ACTIVE_RUNTIME", None)
        if active_runtime is None:
            raise RuntimeError("No active runtime patched. Call patch_agents_with_runtime first.")
        return active_runtime.invoke(prompt, temperature=temperature)

    agents.invoke_with_fallback = local_invoke_with_fallback
    return agents


def clear_runtime_patch():
    import src.agents as agents
    if hasattr(agents, "_ACTIVE_RUNTIME"):
        agents._ACTIVE_RUNTIME = None


def evaluate_company_agentic(runtime_dict, samples):
    clear_runtime_patch()
    patch_agents_with_runtime(AgenticRuntime(runtime_dict))
    try:
        return evaluator.run_evaluation(samples)
    finally:
        clear_runtime_patch()


print("QueryMate agentic bridge ready (provider-free shim mode).")

🔧 Initializing RAG Schema Vectorstore...
✅ RAG System Ready!
QueryMate agentic bridge ready (provider-free shim mode).


## 5.5) QueryMate Agentic Bridge (Same Evaluation Pattern as First Notebook)

This section patches QueryMate agent runtime invocation and uses `tests/evaluator.py` for company-domain execution accuracy, matching the first notebook's evaluation style.

## 6) Baseline Evaluation on Company

In [8]:
baseline_company_eval = company_test[:BASELINE_EVAL_COMPANY_N]

print("Loading baseline runtime...")
baseline_runtime = load_runtime(BASE_MODEL_ID)

print("Running baseline on company with QueryMate agentic evaluator...")
baseline_company_report = evaluate_company_agentic(
    baseline_runtime,
    baseline_company_eval,
)

print("Baseline company execution accuracy (relaxed):", baseline_company_report.get("execution_accuracy"))
print("Baseline company execution accuracy (strict):", baseline_company_report.get("strict_execution_accuracy"))

# Release baseline runtime before training.
del baseline_runtime
clear_cuda()

Loading baseline runtime...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Running baseline on company with QueryMate agentic evaluator...

────────────────────────────────────────────────────────────
  QueryMate Golden Set Evaluation  (50 queries)
────────────────────────────────────────────────────────────

[01/50] Q802 (medium) — Show manager and department for employee Ayaan Saxena 553....
     ✗  FAIL  (27.6s)
          Expected (1 rows): [('IT Manager 1', 'IT')]
          Got      (0 rows): []
          Gen SQL: SELECT T1.ManagerName, T1.DeptName
FROM Departments AS T1
JOIN Employees AS T2 ON T1.DeptID = T2.DeptID
WHERE T2.Name = 
[02/50] Q473 (hard) — Using EXCEPT, list employees who have A/B Testing but not Docker....
     ✗  FAIL  (26.8s)
          Expected (65 rows): [('Aisha Bhalla 568',), ('Aisha Naidu 895',), ('Anaya Ghosh 83',)]...
          Got      (0 rows): []
          Gen SQL: SELECT e.Name
FROM Employees e
JOIN EmployeeProjects ep ON e.EmpID = ep.EmpID
WHERE ep.ProjectID IN (
    SELECT Project
[03/50] Q599 (hard) — Using EXCEPT, list empl

## 7) Training Utilities for QLoRA Adapter Fine-Tuning

In [9]:
def format_company_train_text(item: dict) -> str:

    prompt = company_prompt(item["question"])

    return f"{prompt}\n{item['sql']}"





def format_spider_train_text(item: dict) -> str:

    prompt = spider_prompt(item)

    gold_sql = item.get("query", "")

    return f"{prompt}\n{gold_sql}"





def make_text_dataset(samples, formatter):

    rows = [{"text": formatter(s)} for s in samples]

    return Dataset.from_list(rows)





def train_adapter(

    adapter_name: str,

    train_samples,

    val_samples,

    formatter,

    output_dir: Path,

    num_train_epochs: int = 1,

    max_steps: int = -1,

    learning_rate: float = 5e-5,

):

    output_dir = Path(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)



    compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

    bnb_config = BitsAndBytesConfig(

        load_in_4bit=True,

        bnb_4bit_quant_type="nf4",

        bnb_4bit_compute_dtype=compute_dtype,

        bnb_4bit_use_double_quant=True,

    )



    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

    if tokenizer.pad_token is None:

        tokenizer.pad_token = tokenizer.eos_token



    model = AutoModelForCausalLM.from_pretrained(

        BASE_MODEL_ID,

        quantization_config=bnb_config,

        device_map="auto",

        trust_remote_code=True,

    )

    model.config.use_cache = False

    model.gradient_checkpointing_enable()

    model = prepare_model_for_kbit_training(model)



    lora_config = LoraConfig(

        r=16,

        lora_alpha=32,

        lora_dropout=0.05,

        bias="none",

        task_type="CAUSAL_LM",

        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],

    )

    model = get_peft_model(model, lora_config)

    model.print_trainable_parameters()



    train_ds = make_text_dataset(train_samples, formatter)

    val_ds = make_text_dataset(val_samples, formatter) if val_samples else None



    def tok(batch):

        return tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LEN)



    train_tok = train_ds.map(tok, batched=True, remove_columns=["text"])

    val_tok = val_ds.map(tok, batched=True, remove_columns=["text"]) if val_ds is not None else None



    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)



    args_kwargs = {

        "output_dir": str(output_dir),

        "per_device_train_batch_size": 1,

        "gradient_accumulation_steps": 8,

        "learning_rate": learning_rate,

        "num_train_epochs": num_train_epochs,

        "max_steps": max_steps,

        "logging_steps": 10,

        "save_strategy": "epoch",

        "optim": "paged_adamw_8bit",

        "gradient_checkpointing": True,

        "report_to": "none",
        "disable_tqdm": False,

        "fp16": compute_dtype == torch.float16,

        "bf16": compute_dtype == torch.bfloat16,

    }



    has_eval = val_tok is not None and len(val_tok) > 0

    ta_params = inspect.signature(TrainingArguments.__init__).parameters

    if has_eval:

        if "evaluation_strategy" in ta_params:

            args_kwargs["evaluation_strategy"] = "epoch"

        elif "eval_strategy" in ta_params:

            args_kwargs["eval_strategy"] = "epoch"

        args_kwargs["per_device_eval_batch_size"] = 1



    train_args = TrainingArguments(**args_kwargs)



    trainer = Trainer(

        model=model,

        args=train_args,

        train_dataset=train_tok,

        eval_dataset=val_tok if has_eval else None,

        data_collator=collator,

    )



    print(f"Training adapter: {adapter_name}")

    trainer.train()



    final_adapter = output_dir / "final_adapter"

    trainer.model.save_pretrained(final_adapter)

    tokenizer.save_pretrained(final_adapter)



    del trainer

    del model

    clear_cuda()



    print(f"Saved adapter [{adapter_name}] to: {final_adapter}")

    return final_adapter

In [10]:
# T4-focused training defaults (company-only)
COMPANY_TRAIN_EPOCHS = 2
COMPANY_LEARNING_RATE = 1e-4
COMPANY_MAX_STEPS = -1

print("T4 profile:")
print({
    "company": {"epochs": COMPANY_TRAIN_EPOCHS, "lr": COMPANY_LEARNING_RATE},
    "note": "Avoid LR=5e-1; it is too high for QLoRA fine-tuning.",
})

T4 profile:
{'company': {'epochs': 2, 'lr': 0.0001}, 'note': 'Avoid LR=5e-1; it is too high for QLoRA fine-tuning.'}


### T4 QLoRA Hyperparameter Profile

Recommended for stable training on T4:
- `5e-5` is a safe QLoRA learning rate for company adapter
- `5e-5` to `1e-4` is a common range for Spider generalized adapter
- `5e-1` is far too high and will diverge

Default profile below is tuned for stability first, then quality.

## 8) Fine-Tune Company Adapter, Then Evaluate

In [11]:
company_adapter_dir = train_adapter(
    adapter_name="company",
    train_samples=company_train,
    val_samples=company_val,
    formatter=format_company_train_text,
    output_dir=OUTPUT_ROOT / "company_adapter",
    num_train_epochs=COMPANY_TRAIN_EPOCHS,
    max_steps=COMPANY_MAX_STEPS,
    learning_rate=COMPANY_LEARNING_RATE,
)

company_runtime = load_runtime(BASE_MODEL_ID, adapter_path=company_adapter_dir)
company_ft_eval_set = company_test[:FT_EVAL_COMPANY_N]

company_ft_company_report = evaluate_company_agentic(
    company_runtime,
    company_ft_eval_set,
)

print("Company adapter on company execution accuracy (relaxed):", company_ft_company_report.get("execution_accuracy"))
print("Company adapter on company execution accuracy (strict):", company_ft_company_report.get("strict_execution_accuracy"))

del company_runtime
clear_cuda()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Training adapter: company


Epoch,Training Loss,Validation Loss
1,0.026159,0.024837
2,0.021518,0.020695


Saved adapter [company] to: /content/outputs/dual_adapters/company_adapter/final_adapter


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
  QueryMate Golden Set Evaluation  (50 queries)
────────────────────────────────────────────────────────────

[01/50] Q802 (medium) — Show manager and department for employee Ayaan Saxena 553....
     ✗  FAIL  (45.0s)
          Expected (1 rows): [('IT Manager 1', 'IT')]
          Got      (1 rows): [('IT', 'Rohan Gupta')]
          Gen SQL: SELECT d.DeptName, d.ManagerName FROM Departments d WHERE d.DeptID = (SELECT DeptID FROM Employees WHERE Name = 'Ayaan S
[02/50] Q473 (hard) — Using EXCEPT, list employees who have A/B Testing but not Docker....
     ✓  PASS  (45.7s)  rows=65
[03/50] Q599 (hard) — Using EXCEPT, list employees who have Docker but not NLP....
     ✓  PASS  (40.4s)  rows=62
[04/50] Q215 (easy) — How many employees joined on or after 2024-01-01?...
     ✓  PASS  (30.7s)  rows=1
[05/50] Q100 (hard) — For the Operations department, show managers and their number of ...
     ✓  PASS  (54.0s)  rows=2
[06/50] Q34

## 9) Compare Baseline vs Company Adapter

In [12]:
summary = pd.DataFrame([
    {
        "Run": "Baseline (Base Model)",
        "Company Exec (Relaxed)": baseline_company_report.get("execution_accuracy"),
        "Company Exec (Strict)": baseline_company_report.get("strict_execution_accuracy"),
        "Company Partial-Credit": baseline_company_report.get("partial_credit_accuracy"),
        "Company Eval N": baseline_company_report.get("total"),
    },
    {
        "Run": "Company Adapter",
        "Company Exec (Relaxed)": company_ft_company_report.get("execution_accuracy"),
        "Company Exec (Strict)": company_ft_company_report.get("strict_execution_accuracy"),
        "Company Partial-Credit": company_ft_company_report.get("partial_credit_accuracy"),
        "Company Eval N": company_ft_company_report.get("total"),
    },
])

display(summary)
print("Company adapter path:", company_adapter_dir)

,Run,Company Exec (Relaxed),Company Exec (Strict),Company Partial-Credit,Company Eval N
0,Baseline (Base Model),0.3830,0.3191,0.3511,50
1,Company Adapter,0.7778,0.7333,0.7556,50


Company adapter path: /content/outputs/dual_adapters/company_adapter/final_adapter
